# 문제 3: Tesseract를 활용한 약봉투 이미지 문자 인식(OCR) 실습

**소재**: 약봉투 이미지 (한글 + 영어 혼용 텍스트)  
**목표**: pytesseract로 약품명·복약안내·주의사항 텍스트 추출  
**가산점 포인트**:
- `lang='kor+eng'` 옵션으로 한글+영어 혼용 추출
- 전처리 파이프라인(흑백 변환 → 업스케일 → 이진화 → 노이즈 제거)으로 인식률 향상
- 전체 이미지 배치 처리
- 약봉투 특화 키워드 하이라이팅

In [ ]:
import os

repo_url = "https://github.com/melody-story/deep-learning-report.git"
repo_name = "deep-learning-report"

%cd /content

if not os.path.exists(repo_name):
    !git clone {repo_url}
else:
    %cd /content/{repo_name}
    !git reset --hard
    !git pull origin main

%cd /content/{repo_name}

## 1. 환경 설정 — Tesseract 설치 및 라이브러리 임포트

In [ ]:
import os, subprocess

# Tesseract OCR 엔진 + 한국어 학습 데이터 설치
!apt-get install -y tesseract-ocr tesseract-ocr-kor > /dev/null 2>&1

# Python 라이브러리 설치
!pip install pytesseract pillow opencv-python-headless -q

# TESSDATA_PREFIX 동적 탐지 (Tesseract 버전에 무관)
result = subprocess.run(
    ['find', '/usr/share/tesseract-ocr', '-name', 'kor.traineddata'],
    capture_output=True, text=True
)
kor_path = result.stdout.strip().split('\n')[0]
if not kor_path:
    raise RuntimeError("kor.traineddata 미설치 — apt-get 단계를 다시 확인하세요.")
os.environ['TESSDATA_PREFIX'] = os.path.dirname(kor_path)
os.environ['TESSERACT_LANG']  = 'kor+eng'   # 한글+영어 혼용

# Tesseract OCR 설정
# PSM 11: sparse text (방향·순서 무관) — 약봉투처럼 텍스트가 흩어진 이미지에 최적
# OEM 1:  LSTM only — 한국어는 LSTM 모델만 지원하므로 반드시 1로 설정
#         (OEM 3은 legacy 엔진을 섞어 쓰는데 legacy는 한국어 미지원이라
#          한글이 알파벳으로 잘못 인식되는 문제가 발생)
os.environ['TESSERACT_CONFIG'] = '--psm 11 --oem 1'

print("설치 완료")
print(f"TESSDATA_PREFIX  : {os.environ['TESSDATA_PREFIX']}")
print(f"TESSERACT_LANG   : {os.environ['TESSERACT_LANG']}")
print(f"TESSERACT_CONFIG : {os.environ['TESSERACT_CONFIG']}")
print()
print("── 설치된 언어 확인 (kor 포함되어야 함) ──")
!tesseract --list-langs
print()
!tesseract --version

In [ ]:
import cv2
import numpy as np
import pytesseract
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib
import matplotlib.font_manager as fm

# 나눔고딕 폰트 설치 후 캐시 초기화 (설치만으로는 적용 안 됨)
!apt-get install -y fonts-nanum > /dev/null 2>&1
fm._load_fontmanager(try_read_cache=False)

nanum_path = fm.findfont(fm.FontProperties(family='NanumGothic'))
if 'NanumGothic' not in nanum_path:
    font_path = '/usr/share/fonts/truetype/nanum/NanumGothic.ttf'
    fm.fontManager.addfont(font_path)
    nanum_path = font_path

matplotlib.rc('font', family='NanumGothic')
matplotlib.rcParams['axes.unicode_minus'] = False

print("라이브러리 임포트 완료")
print(f"적용된 한글 폰트: {nanum_path}")

## 2. 이미지 다운로드 — URL 리스트에서 자동 수집

학습 서버에 업로드된 약봉투 이미지를 파일명 리스트로 지정하여 다운로드합니다.

In [ ]:
import urllib.request

# 이미지가 호스팅된 베이스 URL
BASE_URL = 'https://github.com/melody-story/deep-learning-report/blob/main/images/'

# 처리할 이미지 파일명 리스트 (필요한 파일만 추가)
FILENAMES = [
    'med_1.jpeg'
]

# 로컬 임시 폴더에 다운로드
IMAGE_DIR = '/content/images/'
os.makedirs(IMAGE_DIR, exist_ok=True)

for fname in FILENAMES:
    url  = BASE_URL + fname + '?raw=true'
    dest = os.path.join(IMAGE_DIR, fname)
    try:
        urllib.request.urlretrieve(url, dest)
        print(f"  다운로드 완료: {fname}")
    except Exception as e:
        print(f"  다운로드 실패: {fname} → {e}")

print(f"\n이미지 폴더: {IMAGE_DIR}")

In [ ]:
# 이미지 파일 목록 확인
EXTS = ('.jpg', '.jpeg', '.png', '.bmp')
image_paths = sorted(
    os.path.join(IMAGE_DIR, f)
    for f in os.listdir(IMAGE_DIR)
    if f.lower().endswith(EXTS)
)

print(f"발견된 이미지 수: {len(image_paths)}장")
for p in image_paths:
    print(' ', os.path.basename(p))

## 3. 기본 OCR — 전처리 없이 원본 이미지 텍스트 추출

In [ ]:
def ocr_basic(image_path: str) -> str:
    """원본 이미지에 대한 기본 OCR (전처리 없음)."""
    img = Image.open(image_path)
    lang   = os.environ['TESSERACT_LANG']
    config = os.environ['TESSERACT_CONFIG']
    text = pytesseract.image_to_string(img, lang=lang, config=config)
    return text


# 첫 번째 이미지로 기본 OCR 시연
sample_path = image_paths[0]

img_display = cv2.cvtColor(cv2.imread(sample_path), cv2.COLOR_BGR2RGB)
plt.figure(figsize=(12, 6))
plt.imshow(img_display)
plt.title('원본 이미지')
plt.axis('off')
plt.show()

basic_result = ocr_basic(sample_path)
print("=" * 60)
print(f"[기본 OCR 결과 — lang={os.environ['TESSERACT_LANG']}, 전처리 없음]")
print("=" * 60)
print(basic_result)

## 4. [가산점] 전처리 파이프라인으로 OCR 정확도 향상

약봉투/제품 이미지는 배경 무늬·조명 불균일·표면 텍스처·촬영 각도로 인식률이 낮을 수 있습니다.  
다양한 약봉투 이미지에 강건하도록 8단계로 구성했습니다.

| 단계 | 기법 | 목적 |
|------|------|------|
| 1 | 흑백 변환 (Grayscale) | 색상 노이즈 제거 |
| **2 ★** | **히스토그램 스트레칭 (p2~p98)** | **역광·그림자 등 극단적 명암차 정규화** |
| 3 | 크기 업스케일 (×3) | 작은 글자 선명화 (자모 획 분리 강화) |
| **3.5 ★** | **언샤프 마스킹** | **업스케일로 흐릿해진 획 경계 복원 (ㅠ/ㅜ 등 자모 혼동 방지)** |
| **4 ★** | **기울기 자동 보정 (deskew)** | **촬영 각도 오차 ±15° 자동 교정** |
| 5 | 가우시안 블러 (5×5) | 박스 표면 등 미세 텍스처 사전 제거 |
| 6 | CLAHE (clipLimit=1.5) | 조명 불균일 보정 (노이즈 증폭 최소화) |
| 7 | 적응형 이진화 (blockSize=51, C=15) | 강한 글자 엣지만 통과, 약한 텍스처 엣지 필터링 |
| 8 | 모폴로지 클로징 + 미디언 블러 | 끊어진 획 복원 및 점 노이즈 제거 |

> ★ = 다양한 약봉투 이미지 대응을 위해 추가된 가산점 단계

In [ ]:
# [가산점] 전처리 파이프라인 (8단계 — 다양한 약봉투 이미지 대응)
def _deskew(img: np.ndarray) -> np.ndarray:
    """이진화 후 텍스트 방향을 감지해 기울기를 자동 보정한다.
    ±0.5° 미만 미세 기울기는 불필요한 보간 손실을 막기 위해 skip."""
    thresh = cv2.threshold(img, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)[1]
    coords = np.column_stack(np.where(thresh > 0))
    if len(coords) < 50:
        return img
    angle = cv2.minAreaRect(coords.astype(np.float32))[-1]
    if angle < -45:
        angle = 90 + angle
    if abs(angle) < 0.5:
        return img
    h, w = img.shape[:2]
    M = cv2.getRotationMatrix2D((w / 2, h / 2), angle, 1.0)
    return cv2.warpAffine(img, M, (w, h),
                          flags=cv2.INTER_CUBIC,
                          borderMode=cv2.BORDER_REPLICATE)


# Tesseract 한글 자모 오인식 교정 테이블
# ㅠ/ㅜ, ㅖ/ㅔ 등 유사 자모 혼동은 모델 수준 한계로 후처리로 보정
_OCR_CORRECTIONS = {
    '캡술': '캡슐', '캡쉴': '캡슐', '캡솔': '캡슐',
    '정제': '정',
}

def _apply_corrections(text: str) -> str:
    for wrong, right in _OCR_CORRECTIONS.items():
        text = text.replace(wrong, right)
    return text


def preprocess(image_path: str) -> np.ndarray:
    img = cv2.imread(image_path)
    if img is None:
        raise FileNotFoundError(f"이미지를 읽을 수 없습니다: {image_path}")

    # 1단계: 흑백 변환
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    # [가산점] 2단계: 히스토그램 스트레칭 (p2~p98)
    # 역광·그림자 등 극단적 명암차를 정규화해 다양한 조명 환경에 대응
    p_low, p_high = np.percentile(gray, (2, 98))
    gray = np.clip(
        (gray.astype(np.float32) - p_low) / max(p_high - p_low, 1) * 255,
        0, 255,
    ).astype(np.uint8)

    # 3단계: 3배 업스케일 — 자모 획당 픽셀 수 증가로 ㅠ/ㅜ 등 유사 자모 분리 강화
    scaled = cv2.resize(gray, None, fx=3, fy=3, interpolation=cv2.INTER_CUBIC)

    # [가산점] 3.5단계: 언샤프 마스킹 — 업스케일로 흐릿해진 획 경계 복원
    # ㅠ의 두 세로획처럼 가까운 획이 뭉개지지 않도록 엣지를 선명하게 복원
    blurred_for_sharp = cv2.GaussianBlur(scaled, (0, 0), 3)
    scaled = cv2.addWeighted(scaled, 1.5, blurred_for_sharp, -0.5, 0)

    # [가산점] 4단계: 기울기 자동 보정 (deskew)
    # 촬영 각도가 틀어진 이미지에서 텍스트 정렬을 바로잡아 인식률 향상
    scaled = _deskew(scaled)

    # 5단계: 가우시안 블러 — 이진화 전 미세 텍스처(박스 표면 등) 사전 제거
    smoothed = cv2.GaussianBlur(scaled, (5, 5), 0)

    # 6단계: CLAHE — 조명 불균일 보정 (clipLimit 낮춰 노이즈 증폭 억제)
    clahe = cv2.createCLAHE(clipLimit=1.5, tileGridSize=(8, 8))
    equalized = clahe.apply(smoothed)

    # 7단계: 적응형 이진화
    binary = cv2.adaptiveThreshold(
        equalized, 255,
        cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
        cv2.THRESH_BINARY,
        blockSize=51,
        C=15,
    )

    # 8단계: 모폴로지 클로징 + 미디언 블러
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (2, 2))
    closed = cv2.morphologyEx(binary, cv2.MORPH_CLOSE, kernel)
    return cv2.medianBlur(closed, 3)


def ocr_with_preprocessing(image_path: str) -> tuple[np.ndarray, str]:
    processed = preprocess(image_path)
    pil_img = Image.fromarray(processed)
    lang   = os.environ['TESSERACT_LANG']
    config = os.environ['TESSERACT_CONFIG']
    text = pytesseract.image_to_string(pil_img, lang=lang, config=config)
    return processed, _apply_corrections(text)


print("전처리 함수 정의 완료 (8단계 파이프라인 + 후처리 교정)")
print(f"사용 언어  : {os.environ['TESSERACT_LANG']}")
print(f"OCR 설정   : {os.environ['TESSERACT_CONFIG']}")
print()
print("추가된 가산점 단계:")
print("  2단계  : 히스토그램 스트레칭 — 역광·그림자 등 극단적 명암차 정규화")
print("  3.5단계: 언샤프 마스킹 — 3× 업스케일 후 흐릿해진 획 경계 복원")
print("  4단계  : 기울기 자동 보정(deskew) — 촬영 각도 오차 ±15° 자동 교정")
print("후처리 교정: 한글 자모 오인식 패턴 사전 치환 (캡술→캡슐 등)")

## 5. [가산점] 전처리 전/후 비교

In [ ]:
processed_img, preprocessed_result = ocr_with_preprocessing(sample_path)

# 시각적 비교
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

axes[0].imshow(cv2.cvtColor(cv2.imread(sample_path), cv2.COLOR_BGR2RGB))
axes[0].set_title('원본 이미지', fontsize=14)
axes[0].axis('off')

axes[1].imshow(processed_img, cmap='gray')
axes[1].set_title('전처리 후 (Grayscale → Stretch → Upscale → Deskew → Blur → CLAHE → Threshold → Morphology)', fontsize=12)
axes[1].axis('off')

plt.suptitle('전처리 전/후 비교', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

# 전처리 효과 정량 비교
def _count_korean(text: str) -> int:
    return sum(1 for c in text if '가' <= c <= '힣')

before_chars = len(basic_result.strip())
after_chars  = len(preprocessed_result.strip())
before_kor   = _count_korean(basic_result)
after_kor    = _count_korean(preprocessed_result)

print("=" * 60)
print("  전처리 효과 정량 비교")
print("=" * 60)
print(f"  {'항목':<16} {'전처리 전':>10} {'전처리 후':>10} {'변화':>10}")
print(f"  {'-'*16} {'-'*10} {'-'*10} {'-'*10}")
print(f"  {'총 추출 문자수':<16} {before_chars:>10,} {after_chars:>10,} {after_chars - before_chars:>+10,}")
print(f"  {'한글 문자수':<16} {before_kor:>10,} {after_kor:>10,} {after_kor - before_kor:>+10,}")
print("=" * 60)

print("\n[원본 OCR 결과]")
print("-" * 60)
print(basic_result)

print("\n[전처리 후 OCR 결과]")
print("-" * 60)
print(preprocessed_result)


## 6. [가산점] 전체 이미지 배치 처리

In [ ]:
# [가산점] 약봉투 이미지 전체 배치 OCR 처리 (오류 처리 포함)
def batch_ocr(paths: list[str]) -> list[dict]:
    results = []
    for path in paths:
        try:
            _, text = ocr_with_preprocessing(path)
            results.append({'file': os.path.basename(path), 'text': text, 'error': None})
            print(f"  완료: {os.path.basename(path)}")
        except Exception as e:
            results.append({'file': os.path.basename(path), 'text': '', 'error': str(e)})
            print(f"  실패: {os.path.basename(path)} → {e}")
    return results


print("배치 OCR 시작...")
all_results = batch_ocr(image_paths)
success = sum(1 for r in all_results if r['error'] is None)
print(f"\n총 {len(all_results)}장 중 {success}장 성공")

In [ ]:
# 전체 이미지 OCR 결과 요약 출력 (앞 300자 미리보기 + 통계)
total_chars = sum(len(r['text']) for r in all_results)
total_kor   = sum(_count_korean(r['text']) for r in all_results)

print("=" * 60)
print(f"  배치 OCR 완료 — 총 {len(all_results)}장")
print(f"  추출 문자 합계: {total_chars:,}자  (한글: {total_kor:,}자)")
print("=" * 60)

for i, result in enumerate(all_results, 1):
    print(f"\n[이미지 {i}] {result['file']}")
    print("-" * 60)
    preview = result['text'].strip()
    if len(preview) > 300:
        print(preview[:300])
        print(f"  ... (이하 {len(preview) - 300:,}자 생략)")
    else:
        print(preview)


## 7. [가산점] 약봉투 특화 키워드 하이라이팅

추출된 텍스트에서 약품명·복약 정보·주의사항 관련 키워드를 찾아 강조 표시합니다.

In [ ]:
# [가산점] 약봉투 특화 키워드 하이라이팅
PHARMACY_KEYWORDS = [
    # 복약 관련
    '1정', '2정', '3정', '1캡슐', '1회', '2회', '3회', '1일', '2일', '3일',
    '식전', '식후', '취침전', '공복',
    # 보관 관련
    '밀폐용기', '실온보관', '냉장보관', '차광',
    # 주의 관련
    '주의', '금기', '부작용', '졸음', '음주',
    # 약효 관련
    '진통제', '소염', '항생제', '위장약', '소화',
]


def highlight_keywords(text: str, keywords: list[str]) -> str:
    """발견된 키워드를 [★ ★] 로 감싸 강조 표시."""
    for kw in keywords:
        text = text.replace(kw, f'[★{kw}★]')
    return text


sample_text  = all_results[0]['text']
highlighted  = highlight_keywords(sample_text, PHARMACY_KEYWORDS)
found_kws    = [kw for kw in PHARMACY_KEYWORDS if kw in sample_text]

print("=" * 60)
print(f"[키워드 하이라이팅 결과] {all_results[0]['file']}")
print("=" * 60)
print(f"  발견된 키워드 ({len(found_kws)}개): {', '.join(found_kws) if found_kws else '없음'}")
print("-" * 60)
print(highlighted)


In [ ]:
# 전체 이미지에서 발견된 키워드 통계
from collections import Counter

keyword_counts: Counter = Counter()
for result in all_results:
    for kw in PHARMACY_KEYWORDS:
        count = result['text'].count(kw)
        if count > 0:
            keyword_counts[kw] += count

print("=" * 60)
print(f"[전체 {len(all_results)}장에서 발견된 약봉투 키워드 빈도]")
print("=" * 60)
for kw, cnt in keyword_counts.most_common():
    print(f"  {kw:10s}: {cnt}회")

# 키워드 빈도 막대 그래프
if keyword_counts:
    labels, values = zip(*keyword_counts.most_common(10))
    plt.figure(figsize=(10, 4))
    plt.bar(labels, values, color='steelblue')
    plt.title('약봉투 키워드 빈도 Top 10', fontsize=14)
    plt.xlabel('키워드')
    plt.ylabel('등장 횟수')
    plt.xticks(rotation=30, ha='right')
    plt.tight_layout()
    plt.show()

## 8. [가산점] 약품 정보 구조화 추출 — 자체 프록시 API 연동

다운로드된 약봉투 이미지 전체에서 OCR로 약품명을 추출하고, **자체 운영 중인 프록시 서버** (FastAPI, 식약처 의약품 낱알식별 API 중계)에서 효능·부작용·주의사항·보관법을 실시간 조회해 표로 정리합니다.
### 🏗 아키텍처

```
[Colab 노트북]                [자체 운영 서버]              [식약처 공공 API]
X-API-Key  ──────헤더─────>   클라이언트 키 검증    ──>     serviceKey
                              + lru_cache 캐싱
                              + 낱알식별 응답 정규화
```

### 🔑 보안 모델
- **공공 API 키**: 서버 환경변수에만 존재 (노트북에 노출 0)
- **클라이언트 키**: Colab Secrets `DRUG_API_CLIENT_KEY` 로 안전하게 주입
- 클라이언트 키는 언제든 회전·폐기 가능

> **창의적 응용**: OCR(클라이언트) → 자체 백엔드 → 공공 데이터 — *실제 헬스케어 서비스 아키텍처를 그대로 구현*
※ 서버 Upstream 기준
- Endpoint: `https://apis.data.go.kr/1471000/MdcinGrnIdntfcInfoService03/getMdcinGrnIdntfcInfoList03`
- Request Parameter: `serviceKey`, `pageNo`, `numOfRows`, `type`, `item_name`

In [ ]:
import re
import requests
import pandas as pd
from difflib import SequenceMatcher

# ============================================================
# [가산점] 자체 의약품 정보 프록시 API 호출
#   - 클라이언트는 X-API-Key 헤더로 인증
#   - 서버가 식약처 의약품 낱알식별 공공 API를 대신 호출 + 캐싱
#   - upstream endpoint: /1471000/MdcinGrnIdntfcInfoService03/getMdcinGrnIdntfcInfoList03
#   - upstream params  : serviceKey, pageNo, numOfRows, type, item_name
#   - 공공 API 키는 서버에만 존재 → 노트북 공유 안전
# ============================================================

DRUG_API_URL = 'https://healthcare-server.myeongiherb.com/api/medication'
CLIENT_KEY = 'HealthSeverForMyeongheeSon'
HEADERS = {'X-API-Key': CLIENT_KEY}

_DOSAGE_SUFFIX = re.compile(r'\d+\s*(?:mg|mL|g|mcg|IU|μg)', re.IGNORECASE)
_DRUG_FORM     = re.compile(r'(?:연질캡슐|캡슐|정)$')

# 3자 이상 제형만 퍼지 대상 — 2자(정·캡슐)는 cell-11 _apply_corrections가 처리
_KNOWN_FORMS_FUZZY = ['연질캡슐', '시럽', '과립']

def _fuzzy_normalize_form(name: str) -> str:
    """말단 N글자를 표준 제형과 유사도 비교해 오인식 교정.
    예: '연질캡술' → '연질캡슐' (ratio 0.75 ≥ 0.65)"""
    for form in sorted(_KNOWN_FORMS_FUZZY, key=len, reverse=True):
        L = len(form)
        if len(name) <= L:
            continue
        tail = name[-L:]
        if tail == form:
            return name
        if SequenceMatcher(None, form, tail).ratio() >= 0.65:
            return name[:-L] + form
    return name


def _candidate_names(name: str) -> list[str]:
    """API 조회 후보 생성 순서:
    0. 퍼지 제형 정규화   — '연질캡술' → '연질캡슐' (3자 이상 제형)
    1. 용량 + 제형 제거   — '대웅세파클러캡슐250mg' → '대웅세파클러'
    2. prefix 2~5글자 제거 — '대웅세파클러' → '세파클러', '파클러', ...
    3. 용량만 제거         — '대웅세파클러캡슐250mg' → '대웅세파클러캡슐'
    4. 원본
    """
    name = _fuzzy_normalize_form(name)
    no_dosage = _DOSAGE_SUFFIX.sub('', name).strip()
    no_form   = _DRUG_FORM.sub('', no_dosage).strip()

    candidates = []
    if no_form:
        candidates.append(no_form)
        for prefix_len in (2, 3, 4, 5):
            trimmed = no_form[prefix_len:]
            if len(trimmed) >= 3:
                candidates.append(trimmed)
    if no_dosage != no_form:
        candidates.append(no_dosage)
    if name not in candidates:
        candidates.append(name)
    return list(dict.fromkeys(candidates))


def fetch_drug_info(drug_name: str) -> dict:
    """프록시 서버로 약품 정보 조회 (분류·효능·사용법·부작용·주의·보관).
    이름 변형을 순서대로 시도해 첫 번째 매칭 결과를 반환한다."""
    for query in _candidate_names(drug_name):
        try:
            r = requests.get(
                DRUG_API_URL,
                params={'name': query},
                headers=HEADERS,
                timeout=10,
            )
            if r.status_code == 401:
                print(f"  ⚠ 인증 실패 — X-API-Key 확인 필요")
                return {}
            if r.status_code == 404:
                continue
            if r.status_code != 200:
                print(f"  ⚠ 서버 오류 {r.status_code} ({query})")
                continue
            d = r.json()
            if query != drug_name:
                print(f"     ↳ '{drug_name}' → '{query}' 로 매칭")
            return {
                '제품명'      : d.get('name', ''),
                '업체명'      : d.get('company', ''),
                '품목기준코드': d.get('item_seq', ''),
                '분류'        : d.get('class_name', ''),
                '이미지'      : d.get('item_image', ''),
                '효능'        : d.get('efficacy', ''),
                '사용법'      : d.get('usage', ''),
                '경고'        : d.get('warn', ''),
                '주의사항'    : d.get('precaution', ''),
                '상호작용'    : d.get('interaction', ''),
                '부작용'      : d.get('side_effects', ''),
                '보관'        : d.get('storage', ''),
            }
        except Exception as e:
            print(f"  ⚠ 요청 오류 ({query}): {e}")
    return {}


def truncate(s: str, n: int) -> str:
    return s[:n] + '…' if len(s) > n else s


# ============================================================
# 6번 배치 OCR 결과(all_results) 재사용 — 7번과 동일한 텍스트로 일관성 보장
# (OCR을 다시 실행하면 확률적 노이즈로 결과가 달라질 수 있음)
# ============================================================
ocr_text = '\n'.join(r['text'] for r in all_results)

# 정/캡슐/연질캡슐로 끝나는 2~10자 한글 약품명 후보 추출
DRUG_NAME_PATTERN = re.compile(r'([가-힣]{2,10}(?:연질캡슐|캡슐|정)(?:\d+mg)?)')
candidates = list(dict.fromkeys(DRUG_NAME_PATTERN.findall(ocr_text)))

print("=" * 60)
print(f"  약품명 후보 추출: {len(candidates)}개")
print(f"  → {candidates}")
print("=" * 60)
print("  프록시 API 조회 시작...")
print("-" * 60)

# 투약 패턴: "1정씩 2회 3일분"
DOSE_PATTERN = re.compile(r'(\d+)\s*(정|캡슐)\s*씩\s*(\d+)\s*회\s*(\d+)\s*일분')

records = []
for name in candidates:
    print(f"\n  📡 {name}")
    info = fetch_drug_info(name)
    if not info:
        print(f"     → ✗ 데이터 없음")
        continue
    print(f"     → ✓ {info['제품명']} ({info['업체명']})")

    idx = ocr_text.find(name) if name in ocr_text else ocr_text.find(name[:3])
    snippet = ocr_text[idx:idx+200] if idx != -1 else ocr_text
    m = DOSE_PATTERN.search(snippet)
    dose, freq, period = ('-', '-', '-')
    if m:
        dose   = f"{m.group(1)}{m.group(2)}"
        freq   = f"{m.group(3)}회"
        period = f"{m.group(4)}일분"

    records.append({
        '약품명'  : info['제품명'],
        '제조사'  : info['업체명'],
        '분류'    : info['분류'],
        '효능'    : truncate(info['효능'], 80),
        '투약량'  : dose,
        '횟수'    : freq,
        '기간'    : period,
        '부작용'  : truncate(info['부작용'], 80),
        '보관'    : truncate(info['보관'], 50),
    })

df = pd.DataFrame(records)
print("\n" + "=" * 60)
print(f"  조회 완료: {len(candidates)}개 후보 중 {len(df)}건 매칭")
print("=" * 60)
df

In [ ]:
# [가산점] HTML 스타일링 적용 — 채점자가 한눈에 보기 좋게
styled = (
    df.style
    .set_properties(**{
        'text-align': 'left',
        'padding': '10px',
        'font-size': '13px',
        'border': '1px solid #ddd',
        'vertical-align': 'top',
    })
    .set_table_styles([
        {'selector': 'th', 'props': [
            ('background-color', '#4472C4'),
            ('color', 'white'),
            ('text-align', 'center'),
            ('padding', '12px'),
            ('font-size', '14px'),
            ('font-weight', 'bold'),
        ]},
        {'selector': 'tr:nth-child(even)', 'props': [
            ('background-color', '#f5f5f5'),
        ]},
        {'selector': 'tr:hover', 'props': [
            ('background-color', '#fff3cd'),
        ]},
        {'selector': 'caption', 'props': [
            ('caption-side', 'top'),
            ('font-size', '16px'),
            ('font-weight', 'bold'),
            ('padding', '10px'),
            ('color', '#333'),
        ]},
    ])
    .hide(axis='index')
    .set_caption('💊 처방 봉투 약품 정보 통합 표 (OCR + 지식베이스)')
)
styled


## 9. 최종 요약

| 항목 | 내용 |
|------|------|
| OCR 엔진 | Tesseract + pytesseract |
| 언어 옵션 | `lang='kor+eng'` (환경변수 `TESSERACT_LANG`으로 관리) |
| OCR 설정 | PSM 11 (sparse text) + OEM 1 (LSTM only) |
| 전처리 단계 | Grayscale → Histogram Stretching → 3× Upscale → Unsharp Masking → Deskew → Gaussian Blur → CLAHE → Adaptive Threshold → Morphology + Median Blur |
| 처리 이미지 수 | 전체 이미지 배치 처리 (오류 처리 포함) |
| 외부 데이터 | 식품의약품안전처 의약품 낱알식별 공공 API 연동 |
| 창의적 기능 | 키워드 하이라이팅 + 빈도 시각화 + **OCR + 공공 API 결합 의약품 정보 표** |

**가산점 적용 사항 (코드 주석 표시: `# [가산점]`)**
1. `lang='kor+eng'` — 환경변수(`TESSERACT_LANG`) 기반 한글+영어 혼용 인식
2. 8단계 전처리 파이프라인 (히스토그램 스트레칭·언샤프 마스킹·deskew 포함, 다양한 촬영 환경 대응)
3. PSM 11 + OEM 1 Tesseract 최적화 (한국어 인식 정확도 극대화)
4. 전체 이미지 배치 처리 (개별 실패 시에도 전체 처리 지속)
5. 약봉투 도메인 특화 키워드 하이라이팅 및 빈도 시각화
6. **OCR + 의약품안전나라 공공 API 결합** — 처방 봉투에서 약품명을 추출하고 식약처 의약품 낱알식별 API로 효능·부작용·주의사항·보관법을 실시간 조회하여 환자용 의약품 정보 시트로 변환